## 프롬프트 엔지니어링 실습

**프롬프트 엔지니어링이란?**
- AI 모델로부터 원하는 결과를 얻기 위해 **효과적인 질문/명령**을 설계하는 기술
- 같은 질문이라도 **프롬프트를 어떻게 작성하느냐**에 따라 응답의 품질이 크게 달라짐
- 데이터 분석가에게 필수적인 스킬: 데이터 해석, 인사이트 도출, 보고서 작성 등에 활용


**프롬프트 엔지니어링 참고 자료:**
- [Gemini Prompt Engineering](https://ai.google.dev/gemini-api/docs/prompting-strategies?hl=ko)
- [OpenAI Prompt Engineering](https://platform.openai.com/docs/guides/prompt-engineering)
- [Anthropic Prompt Engineering](https://docs.anthropic.com/claude/docs/prompt-engineering)

In [ ]:
# 라이브러리 설치
!pip install google-genai python-dotenv pandas pydantic tqdm

In [1]:
# API 접근을 위한 설정
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pprint import pprint 

# .env 파일에서 API 키 로드
load_dotenv()
api_key = os.getenv('GEMINI_API_KEY')
gemini_model = os.getenv('GEMINI_MODEL', 'gemini-2.5-flash-lite')

# API 키 유효성 검사
api_key_valid = api_key and 'YOUR_API_KEY' not in api_key
print(f"API 키 설정 확인: {'✓' if api_key_valid else '✗'}")
if not api_key_valid:
    print("⚠️  .env 파일에서 GEMINI_API_KEY를 실제 API 키로 설정해주세요!")
print(f"모델 확인: {gemini_model}")


# 클라이언트 초기화
client = genai.Client(api_key=api_key)

# GenerateContentConfig: 생성 모델의 동작 방식을 세밀하게 제어하는 설정
generation_config = types.GenerateContentConfig(
    temperature=0.7,       # 기본값: 1.0 (창의성 조절, 0.0~2.0)
    top_p=0.95,            # 기본값: 0.95 (다양성 조절, 0.0~1.0)
    top_k=64,              # 토큰 선택 범위
    max_output_tokens=100, # 최대 출력 토큰 수
    candidate_count=1,     # 기본값: 1 (응답 생성 수)
    system_instruction="너는 AI 기술을 쉽게 설명하는 전문가야. 초보자도 이해할 수 있게 설명해줘."
)

API 키 설정 확인: ✓
모델 확인: gemini-2.5-flash-lite


 ### 1. 명확한 지침 제공 (Clear Instructions)



 **핵심 원칙**

 - 중요한 세부사항, 문맥, 제약조건을 명시

 - 모호한 표현 대신 구체적인 요구사항 전달

 - 출력 형식과 길이 지정

In [2]:
print("=" * 80)
print("📌 실습 1: 명확성의 중요성 - 데이터 분석 시나리오")
print("=" * 80)

# ❌ 나쁜 예시: 모호한 프롬프트
bad_prompt = "데이터 분석에 대해 설명해줘"

print(f"\n❌ 나쁜 프롬프트:\n{bad_prompt}\n")
print("🤖 AI 응답:")

print("\n" + "-" * 80 + "\n")

response = client.models.generate_content(
    model=gemini_model,
    contents=bad_prompt,
    config=generation_config
)
pprint(response.text)


# ✅ 좋은 예시: 명확하고 구체적인 프롬프트
good_prompt = """다음 조건에 맞춰 A/B 테스트 결과를 분석하는 방법을 설명해줘:

**상황:**
- 이커머스 사이트의 '구매하기' 버튼 색상 변경 테스트 (빨강 vs 파랑)
- A그룹(빨강): 1000명, 전환율 5.2%
- B그룹(파랑): 1000명, 전환율 6.8%

**요구사항:**
1. 통계적 유의성 판단 방법 (카이제곱 검정 기준)
2. 비즈니스 의사결정을 위한 해석 (2-3문장)
3. 주의사항 1가지

**제약:**
- 전체 5문장 이내
- 전문용어 사용 시 괄호 안에 쉬운 설명 추가
"""

print(f"\n✅ 좋은 프롬프트:\n{good_prompt}\n")
print("🤖 AI 응답:")

response = client.models.generate_content(
    model=gemini_model,
    contents=good_prompt,
    config=generation_config
)
pprint(response.text)

📌 실습 1: 명확성의 중요성 - 데이터 분석 시나리오

❌ 나쁜 프롬프트:
데이터 분석에 대해 설명해줘

🤖 AI 응답:

--------------------------------------------------------------------------------

('## 데이터 분석, 어렵지 않아요! 🤔\n'
 '\n'
 '안녕하세요! AI 기술 전문가로서, 오늘은 많은 분들이 궁금해하시는 **데이터 분석**에 대해 아주 쉽고 재미있게 설명해 드릴게요. 마치 '
 '요리사가 맛있는 음식을 만들기 위해 재료를 다듬고 조리하는 것처럼, 데이터 분석도 **데이터라는 재료를 가지고 유용한 정보라는 맛있는 '
 '결과물을 만들어내는 과정**이라고 생각하면 쉬워요.\n'
 '\n'
 '### 1. 데이터, 그')

✅ 좋은 프롬프트:
다음 조건에 맞춰 A/B 테스트 결과를 분석하는 방법을 설명해줘:

**상황:**
- 이커머스 사이트의 '구매하기' 버튼 색상 변경 테스트 (빨강 vs 파랑)
- A그룹(빨강): 1000명, 전환율 5.2%
- B그룹(파랑): 1000명, 전환율 6.8%

**요구사항:**
1. 통계적 유의성 판단 방법 (카이제곱 검정 기준)
2. 비즈니스 의사결정을 위한 해석 (2-3문장)
3. 주의사항 1가지

**제약:**
- 전체 5문장 이내
- 전문용어 사용 시 괄호 안에 쉬운 설명 추가


🤖 AI 응답:
('A/B 테스트 결과 분석 방법을 설명해 드릴게요.\n'
 '\n'
 '1.  **통계적 유의성 판단:** 카이제곱 검정(두 그룹 간 비율 차이가 우연인지 실제인지 통계적으로 확인하는 방법)을 통해 파란색 '
 '버튼의 전환율이 빨간색 버튼보다 통계적으로 유의미하게 높다고 판단할 수 있습니다.\n'
 '2.  **비즈니스 의사결정:** 파란색 버튼이 전환율을 ')


 ### 2. 페르소나 기반 역할 부여 (Role Prompting)



 **핵심 개념**

 - AI에게 특정 전문가 역할을 부여하여 답변의 품질과 관점 조정

 - 데이터 분석가, 통계학자, 비즈니스 컨설턴트 등 역할 지정

In [3]:
print("\n" + "=" * 80)
print("📌 실습 2: 역할 부여 - 같은 데이터, 다른 관점")
print("=" * 80)

# 시나리오: 고객 이탈률 데이터 분석
data_context = """
**고객 이탈 데이터:**
- 전체 고객: 10,000명
- 이탈 고객: 1,200명 (12%)
- 주요 이탈 사유: 가격(45%), 경쟁사 이동(30%), 서비스 불만(25%)
- 평균 구독 기간: 이탈 고객 8개월 vs 유지 고객 24개월
"""

# 역할: 데이터 분석가
system_instruction = """
당신은 3년 차 데이터 분석가입니다.

다음 형식으로 분석 결과를 제공하세요:
1. 핵심 지표 3가지 해석
2. 데이터 기반 개선 방안 2가지
3. 추가 분석이 필요한 항목 1가지

6문장 이내로 작성하고, 숫자와 비율을 활용하여 구체적으로 설명하세요.
"""

user_prompt = f"""
다음 고객 이탈 데이터를 분석하고 인사이트를 도출하세요.

{data_context}
"""

print("👤 역할: 데이터 분석가")
print("🤖 AI 응답:")

generation_config = types.GenerateContentConfig(
    temperature=0.7,
    top_p=0.9,
    max_output_tokens=1000,
    system_instruction=system_instruction
)

response = client.models.generate_content(
    model=gemini_model,
    contents=user_prompt,
    config=generation_config
)
print(response.text)


📌 실습 2: 역할 부여 - 같은 데이터, 다른 관점
👤 역할: 데이터 분석가
🤖 AI 응답:
## 고객 이탈 데이터 분석 결과

**1. 핵심 지표 3가지 해석**

*   **이탈률 12%**: 전체 고객 10,000명 중 1,200명이 이탈하여 12%의 이탈률을 보이고 있습니다. 이는 상당한 규모의 고객 손실이며, 비즈니스 성장에 부정적인 영향을 미칠 수 있습니다.
*   **주요 이탈 사유**: 이탈 고객의 45%가 가격 문제로 이탈했으며, 30%는 경쟁사 이동, 25%는 서비스 불만으로 분석되었습니다. 특히 가격이 가장 큰 이탈 요인으로 작용하고 있음을 알 수 있습니다.
*   **평균 구독 기간**: 이탈 고객의 평균 구독 기간은 8개월인 반면, 유지 고객은 24개월입니다. 이는 고객이 장기적으로 서비스를 이용할수록 이탈 가능성이 낮아짐을 시사합니다.

**2. 데이터 기반 개선 방안 2가지**

*   **가격 경쟁력 강화**: 이탈 사유 1위인 가격 문제를 해결하기 위해, 경쟁사 대비 합리적인 가격 정책을 검토하거나, 장기 구독 고객을 위한 할인 프로모션을 강화해야 합니다. 예를 들어, 1년 이상 구독 시 15% 할인 혜택을 제공하는 방안을 고려할 수 있습니다.
*   **고객 충성도 프로그램 도입**: 평균 구독 기간 차이를 고려하여, 신규 고객의 초기 이탈을 방지하고 장기 고객으로 전환을 유도하는 로열티 프로그램을 설계해야 합니다. 구독 기간별 차등 혜택 제공, 친구 추천 시 보상 지급 등을 통해 고객 참여를 높일 수 있습니다.

**3. 추가 분석이 필요한 항목 1가지**

*   **경쟁사 이동 고객 심층 분석**: 경쟁사로 이동한 30%의 고객들이 어떤 경쟁사로 이동했으며, 해당 경쟁사의 어떤 서비스 또는 가격 정책에 매력을 느꼈는지 구체적으로 파악하는 추가 분석이 필요합니다. 이를 통해 우리 서비스의 약점을 명확히 파악하고 경쟁 우위를 확보할 전략을 수립할 수 있습니다.


 ### 3. 구조화된 구분자 활용 (Delimiter Usage)



 **핵심 개념**

In [4]:
print("\n" + "=" * 80)
print("📌 실습 3: 구분자를 활용한 데이터 요약")
print("=" * 80)

delimiter_prompt = """
다음 <데이터>를 분석하여 요약 리포트를 작성하세요.

<데이터>
2024년 1분기 매출 데이터 분석 결과입니다. 
총 매출은 전년 대비 23% 증가한 150억원을 기록했습니다.
제품별로는 A제품이 60억(40%), B제품이 45억(30%), C제품이 45억(30%)을 차지했습니다.
지역별로는 서울 70억(47%), 경기 50억(33%), 기타 30억(20%)으로 수도권이 80%를 차지합니다.
온라인 채널 비중이 전년 45%에서 55%로 증가하며 디지털 전환이 가속화되고 있습니다.
고객 만족도는 평균 4.2/5.0점으로 전년 대비 0.3점 상승했습니다.
</데이터>

<출력형식>
## 📊 핵심 지표
- 항목: 수치 (해석)

## 💡 주요 인사이트
1. 인사이트 1
2. 인사이트 2
3. 인사이트 3

## ⚠️ 주의점
- 주의사항
</출력형식>

제약: 전체 8문장 이내, 각 인사이트는 1-2문장
"""

print("🤖 AI 응답:")

response = client.models.generate_content(
    model=gemini_model,
    contents=delimiter_prompt,
    config=generation_config
)
print(response.text)


📌 실습 3: 구분자를 활용한 데이터 요약
🤖 AI 응답:
## 📊 핵심 지표
- 총 매출: 150억원 (전년 대비 23% 증가)
- 수도권 매출 비중: 80% (서울 47%, 경기 33%)
- 온라인 채널 매출 비중: 55% (전년 45% 대비 10%p 증가)

## 💡 주요 인사이트
1.  **수도권 집중 현상 완화 필요**: 전체 매출의 80%가 수도권에 집중되어 있어, 지방 시장 공략을 통한 성장 잠재력 확보가 중요합니다.
2.  **온라인 채널 성장세 가속화**: 온라인 채널 비중이 10%p 증가하며 디지털 전환이 성공적으로 이루어지고 있으며, 이에 대한 투자를 지속해야 합니다.
3.  **고객 만족도 상승**: 평균 4.2/5.0점으로 전년 대비 0.3점 상승한 것은 긍정적이며, 이를 유지 및 향상시키기 위한 노력이 필요합니다.

## ⚠️ 주의점
-   **제품별 매출 격차 분석**: A제품이 전체 매출의 40%를 차지하지만, B, C 제품과의 격차를 줄이기 위한 전략 수립이 필요합니다.


 ### 4. Few-Shot Learning (예시 기반 학습)



 **핵심 개념**

 - 원하는 출력 형식의 예시를 2-3개 제공

 - AI가 패턴을 학습하여 동일한 형식으로 응답

 - 일관된 보고서나 문서 작성에 매우 효과적

In [5]:
print("\n" + "=" * 80)
print("📌 실습 4: 감정 분석 인사이트 카드 생성")
print("=" * 80)

system_instruction = """
당신은 소셜 미디어 감정 분석 전문가입니다.

다음 형식으로 감정 분석 결과를 '인사이트 카드' 형태로 작성해주세요:

인사이트: [핵심 발견 사항을 한 문장으로]
데이터: [구체적인 수치와 변화]
해석: [감정 변화의 원인 또는 의미]
액션: [실행 가능한 대응 방안 2가지]

간결하고 명확하게 작성하세요.
"""

few_shot_prompt = """
[예시 1]
인사이트: 신제품 출시 후 긍정 감정 급증
데이터: 긍정 댓글 비율 45% → 72%로 증가 (출시 2주 후), "만족", "추천" 키워드 3배 증가
해석: 신제품에 대한 고객 기대가 실제 경험으로 충족되며 긍정적 입소문 확산
액션: 긍정 리뷰 활용한 SNS 마케팅 강화, 만족 고객 대상 리워드 프로그램 런칭

[예시 2]
인사이트: 배송 관련 부정 감정 집중 발견
데이터: 부정 댓글 중 62%가 배송 이슈, "늦음", "지연" 키워드가 전월 대비 40% 증가
해석: 물류 프로세스 문제로 고객 불만 누적, 재구매 의향 저하 우려
액션: 배송 프로세스 긴급 점검 및 개선, 지연 고객 대상 보상 정책 수립

[새로운 데이터 분석 요청]
데이터: 영화 리뷰 댓글 1,000개 분석 결과
- 긍정: 680건 (68%), 주요 키워드: "감동", "명작", "최고의 연기"
- 부정: 220건 (22%), 주요 키워드: "지루함", "예측 가능", "과대평가"
- 중립: 100건 (10%)
- 전작 대비 긍정 비율 15%p 상승
"""

print("📊 분석할 데이터:")
print("영화 리뷰 감정 분석 결과 (1,000개 댓글)")
print("\n🤖 AI 응답:")

generation_config = types.GenerateContentConfig(
    temperature=0.7,
    top_p=0.9,
    max_output_tokens=400,
    system_instruction=system_instruction
)

response = client.models.generate_content(
    model=gemini_model,
    contents=few_shot_prompt,
    config=generation_config
)
print(response.text)


📌 실습 4: 감정 분석 인사이트 카드 생성
📊 분석할 데이터:
영화 리뷰 감정 분석 결과 (1,000개 댓글)

🤖 AI 응답:
인사이트: 영화에 대한 전반적인 긍정적 반응과 호평 증가
데이터: 긍정 리뷰 68% (전작 대비 15%p 상승), "감동", "명작", "최고의 연기" 키워드 빈도 높음
해석: 전작 대비 향상된 스토리텔링과 배우들의 뛰어난 연기가 관객들에게 깊은 인상을 주며 긍정적인 평가를 이끌어냄
액션: 긍정적인 리뷰를 활용한 홍보 강화 (예: "명작", "최고의 연기" 등 키워드 강조), 배우 인터뷰 및 비하인드 스토리 공개로 관심 증폭


 ### 5. Chain-of-Thought (단계별 사고 유도)



 **핵심 개념**

 - 복잡한 분석 문제를 단계별로 분해

 - AI가 중간 추론 과정을 명시하도록 유도

 - 논리적 오류 감소, 투명한 의사결정 지원

In [ ]:
print("\n" + "=" * 80)
print("📌 실습 5: 복잡한 비즈니스 문제 해결")
print("=" * 80)

cot_prompt = """
당신은 시니어 데이터 분석가입니다. 다음 비즈니스 문제를 단계별로 분석하세요.

**상황:**
온라인 쇼핑몰의 월 매출이 3개월 연속 감소하고 있습니다.
- 1월: 10억 → 2월: 9.5억 (-5%) → 3월: 8.8억 (-7.4%)
- 방문자 수는 유지 (월 50만명)
- 전환율: 4% → 3.5% → 3.2% 
- 평균 객단가: 50,000원 → 54,000원 → 55,000원
- 신규 고객 비율: 60% → 55% → 48%

**분석 프로세스:**

STEP 1: 데이터 해석
- 각 지표의 변화 추이와 의미를 분석하세요.

STEP 2: 근본 원인 가설
- 매출 감소의 가능한 원인 3가지를 제시하고, 각각을 데이터로 뒷받침하세요.

STEP 3: 추가 확인 필요 사항
- 원인을 정확히 파악하기 위해 추가로 확인해야 할 데이터 2가지

STEP 4: 해결 방안
- 가장 가능성 높은 원인에 대한 해결책 2가지 (우선순위 순)

각 STEP을 명확히 구분하여 작성하고, 전체 10문장 이내로 작성하세요.
"""

print("🤖 AI 응답:")

response = client.models.generate_content(
    model=gemini_model,
    contents=cot_prompt,
    config=generation_config
)
print(response.text)


 ### 6. 구조화된 출력 형식 (Structured Output)



 **핵심 개념**

 - JSON, CSV, 마크다운 테이블 등 일관된 형식으로 출력

 - 후처리 및 자동화에 용이

 - 데이터 파이프라인 구축 시 필수

In [6]:
print("\n" + "=" * 80)
print("📌 실습 6: JSON 형식으로 분석 결과 출력")
print("=" * 80)

structured_prompt = """
다음 고객 리뷰 텍스트를 분석하고 JSON 형식으로 결과를 출력하세요.

**리뷰:**
"배송이 생각보다 빨라서 좋았어요! 제품 품질도 만족스럽습니다. 
다만 포장이 조금 부실해서 박스가 찌그러져 왔네요. 
가격 대비 가성비는 좋은 편이라고 생각합니다. 재구매 의향 있어요."

**출력 형식:**
```json
{
  "sentiment": "긍정|중립|부정",
  "sentiment_score": 1-5점,
  "positive_aspects": ["긍정1", "긍정2"],
  "negative_aspects": ["부정1", "부정2"],
  "key_topics": {
    "배송": "평가내용",
    "품질": "평가내용",
    "가격": "평가내용",
    "포장": "평가내용"
  },
  "repurchase_intent": true/false,
  "summary": "한 문장 요약"
}
```

JSON만 출력하고 다른 설명은 추가하지 마세요.
"""

print("🤖 AI 응답:")

response = client.models.generate_content(
    model=gemini_model,
    contents=structured_prompt,
    config=generation_config
)
print(response.text)



📌 실습 6: JSON 형식으로 분석 결과 출력
🤖 AI 응답:
```json
{
  "sentiment": "긍정",
  "sentiment_score": 4,
  "positive_aspects": [
    "빠른 배송",
    "만족스러운 제품 품질",
    "가격 대비 좋은 가성비"
  ],
  "negative_aspects": [
    "부실한 포장으로 인한 박스 찌그러짐"
  ],
  "key_topics": {
    "배송": "생각보다 빨라서 좋았어요",
    "품질": "만족스럽습니다",
    "가격": "가격 대비 가성비는 좋은 편이라고 생각합니다",
    "포장": "포장이 조금 부실해서 박스가 찌그러져 왔네요"
  },
  "repurchase_intent": true,
  "summary": "빠른 배송과 좋은 품질, 가성비에 만족하지만 포장 개선이 필요하며 재구매 의사가 있음."
}
```


 ### 8. 프롬프트 체이닝 (Prompt Chaining)



 **핵심 개념**

 - 복잡한 분석을 여러 단계로 나누어 순차적으로 실행

 - 각 단계의 출력이 다음 단계의 입력이 됨

 - 정확도 향상 및 디버깅 용이

In [7]:
print("\n" + "=" * 80)
print("📌 실습 8: 다단계 고객 세그먼트 분석")
print("=" * 80)

# 원본 데이터
customer_data = """
고객ID, 연령, 구매횟수, 총구매액, 최근구매일
C001, 28, 15, 450000, 2024-03-10
C002, 45, 3, 120000, 2024-01-05
C003, 35, 25, 1200000, 2024-03-15
C004, 52, 8, 340000, 2024-02-20
C005, 31, 18, 680000, 2024-03-12
"""

print("📊 원본 데이터:")
print(customer_data)

# Step 1: 데이터 요약
print("\n" + "-" * 80)
print("STEP 1: 데이터 요약 및 기초 통계")
print("-" * 80)

step1_prompt = f"""
다음 고객 데이터의 기초 통계를 계산하세요:

{customer_data}

출력 항목:
1. 총 고객 수
2. 평균 연령
3. 평균 구매횟수
4. 평균 구매액
5. 최근 30일 이내 구매 고객 수

간단히 숫자만 나열하세요.
"""

response1 = client.models.generate_content(
    model=gemini_model,
    contents=step1_prompt,
    config=generation_config
)
summary_result = response1.text
print("🤖 AI 응답:")
print(summary_result)

# Step 2: 세그먼트 분류
print("\n" + "-" * 80)
print("STEP 2: 고객 세그먼트 분류")
print("-" * 80)

step2_prompt = f"""
앞서 분석한 고객 데이터를 바탕으로 고객을 3개 세그먼트로 분류하세요:

원본 데이터:
{customer_data}

기초 통계:
{summary_result}

분류 기준:
- VIP: 구매횟수 15회 이상 또는 총구매액 60만원 이상
- 일반: 구매횟수 5-14회 또는 총구매액 20-60만원
- 신규: 구매횟수 5회 미만 또는 총구매액 20만원 미만

출력 형식:
### VIP 고객
- 고객ID: 특징

### 일반 고객
- 고객ID: 특징

### 신규 고객
- 고객ID: 특징
"""

response2 = client.models.generate_content(
    model=gemini_model,
    contents=step2_prompt,
    config=generation_config
)
segment_result = response2.text
print("🤖 AI 응답:")
print(segment_result)

# Step 3: 액션 플랜 수립
print("\n" + "-" * 80)
print("STEP 3: 세그먼트별 마케팅 전략")
print("-" * 80)

step3_prompt = f"""
다음 고객 세그먼트 분석 결과를 바탕으로 각 그룹별 마케팅 전략을 제안하세요:

{segment_result}

각 세그먼트별로:
1. 핵심 특징 (1문장)
2. 마케팅 목표
3. 구체적인 액션 2가지

전체 8문장 이내로 작성하세요.
"""

response3 = client.models.generate_content(
    model=gemini_model,
    contents=step3_prompt,
    config=generation_config
)
print("🤖 AI 응답:")
print(response3.text)



📌 실습 8: 다단계 고객 세그먼트 분석
📊 원본 데이터:

고객ID, 연령, 구매횟수, 총구매액, 최근구매일
C001, 28, 15, 450000, 2024-03-10
C002, 45, 3, 120000, 2024-01-05
C003, 35, 25, 1200000, 2024-03-15
C004, 52, 8, 340000, 2024-02-20
C005, 31, 18, 680000, 2024-03-12


--------------------------------------------------------------------------------
STEP 1: 데이터 요약 및 기초 통계
--------------------------------------------------------------------------------
🤖 AI 응답:
1. 5
2. 38.2
3. 13.6
4. 560000
5. 3

--------------------------------------------------------------------------------
STEP 2: 고객 세그먼트 분류
--------------------------------------------------------------------------------
🤖 AI 응답:
## 고객 세그먼트 분석 결과

### VIP 고객
- **C001**: (구매횟수 15회, 총구매액 45만원) - 구매 횟수 기준 충족
- **C003**: (구매횟수 25회, 총구매액 120만원) - 구매 횟수 및 총 구매액 기준 모두 충족
- **C005**: (구매횟수 18회, 총구매액 68만원) - 구매 횟수 및 총 구매액 기준 모두 충족

### 일반 고객
- **C002**: (구매횟수 3회, 총구매액 12만원) - 신규 고객 기준에 더 가까우나, 구매 횟수가 5회 미만이고 총 구매액이 20만원 미만이므로 신규 고객으로 분류하는 것이 더 적합합니다. (수정: 아래 신규 고객으로 재분류)
- **C004**:

## 프롬프트 엔지니어링 핵심 요약

### 5가지 핵심 원칙
1. **명확성**: 모호함 없이 구체적으로
2. **구조화**: 역할-작업-제약 구조 활용
3. **예시 제공**: Few-shot learning으로 일관성 확보
4. **단계별 사고**: 복잡한 문제는 Chain-of-Thought, 프롬프트 체이닝
5. **반복 개선**: 테스트하고 최적화하기


### 기본 구조
```python
system_instruction = """
역할: 당신은 ~입니다
작업: ~을 수행하세요
형식: ~형태로 작성
제약: ~이내, ~개까지
"""

user_prompt = """
실제 분석할 데이터
"""
```

### 💡 핵심 메시지
> "간단하고 명확하게" → 복잡한 프롬프트가 항상 좋은 것은 아님  
> "반복 테스트" → 한 번에 완벽한 프롬프트는 없음  
> "점진적 개선" → 작동하는 프롬프트를 조금씩 개선